# Does detector score track release date *within* the diffusion family? — full AI-GenBench set

`merge_scores.ipynb` §6 fitted one trend across **all** families, which confounds release date
with family (GANs early, diffusion late). This notebook holds family fixed — diffusion only —
and replaces the eyeballed slope with a rank correlation and a p-value.

## What is being asked

Sixteen diffusion generators, released between 2020-06 and 2024-08. For each one, the median
score the panel gives its images. **Does that median move monotonically with release date?**

- **R** (Spearman's rho) — do the two move together in *rank* order? `0` = no relationship,
  `−1` = newest generators get the lowest scores. Rank-based, so no straight-line assumption and
  one outlying generator cannot drive it.
- **p** — how often a correlation at least this strong appears when release dates are shuffled
  at random. Small p = hard to explain as coincidence.

**Why it matters.** Resolving time *within* a family requires score to carry temporal signal
once family is fixed. If R is ~0, a family has to be treated as one bucket and time resolved
only *between* families.

## The unit of analysis: one row per generator (n = 16)

Not one row per image (n = 80,000). All 5,000 Stable Diffusion 1.5 images share one release
date; counting them separately adds no temporal information and shrinks the p-value for
nothing. §5 shows how badly that inflates things.

n = 16 is small no matter how many images each generator has — §4 says which effect sizes this
test can detect at all.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT   = Path.cwd()
SCORES = ROOT / "scores"

DETECTORS = ["cnndetection", "univfd",
             "dmimagedetection_progan", "dmimagedetection_latent",
             "aeroblade"]
zcols = {d: d.replace("dmimagedetection_", "dmid_") for d in DETECTORS}

REAL, FAKE = "#3b6ea5", "#c4462c"
NULLBAND, MUTED = "#d9d9d9", "#888888"

plt.rcParams.update({
    "figure.dpi": 120, "font.size": 9, "axes.grid": True,
    "grid.alpha": 0.25, "grid.linewidth": 0.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 9, "axes.titlelocation": "left", "axes.titleweight": "bold",
})

N_PERM = 200_000   # p-value floor is 1/(N_PERM+1) ~ 5e-6
SEED   = 0

## 1. Rebuild the score table

Same join as `merge_scores.ipynb` §1, repeated here so this notebook stands alone rather than
depending on whether `scores/master_scores.csv` is current.

In [ ]:
man = pd.read_csv(ROOT / "manifest.csv")
M = man.copy()

for d in DETECTORS:
    s = pd.read_csv(SCORES / f"{d}.csv")
    assert list(s.columns) == ["image_id", "raw_score"], f"{d}: unexpected columns"
    M = M.merge(s.rename(columns={"raw_score": d}), on="image_id", how="left")

assert len(M) == len(man), "merge changed the row count"
for d in DETECTORS:
    assert M[d].notna().all(), f"{d}: {M[d].isna().sum()} images unscored"

M["release_date"] = pd.to_datetime(M.release_date, errors="coerce")

# sigma-above-authentic, as in merge_scores.ipynb section 4. This is a per-detector
# monotone linear rescaling, so it cannot change any rank correlation below -- section 2
# asserts that rather than asking you to take it on faith.
Z = M.copy()
for d in DETECTORS:
    ref = M.loc[M.label == 0, d]
    Z[d] = (M[d] - ref.mean()) / ref.std()

print(f"{len(M)} images, {len(DETECTORS)} detectors")

## 2. The diffusion-only table

Sixteen generators, 5,000 images each, one median score per detector.

In [ ]:
D = Z[(Z.label == 1) & (Z.generator_family == "diffusion")]

g = (D.groupby("generator", as_index=False)
      .agg(date=("release_date", "first"), n=("image_id", "size"),
           **{d: (d, "median") for d in DETECTORS})
      .sort_values("date").reset_index(drop=True))

assert g.n.nunique() == 1, "unbalanced images per generator"
assert g.date.is_monotonic_increasing and g.date.nunique() == len(g), "tied release dates"

x  = g.date.map(pd.Timestamp.toordinal).to_numpy(float)
nG = len(g)
print(f"n = {nG} diffusion generators, {g.n.iloc[0]} images each, "
      f"{g.date.min():%Y-%m} to {g.date.max():%Y-%m}")

g.assign(date=g.date.dt.strftime("%Y-%m-%d")).rename(columns=zcols).round(2)

## 3. Spearman R and a permutation p-value

Spearman's rho is Pearson's correlation on ranks. The p-value comes from **shuffling** the
release dates 200,000 times and counting how often chance alone gives a correlation this strong
— no distributional assumption, with a floor of ~5e-6.

The null permutes each detector's own median-ranks, so any ties are carried into it. (On the
local sample AEROBLADE had tied medians; with 5,000 images per generator there are none — the
`ties` column is 0 throughout.)

Five detectors means five tests, so p-values are **Holm**-corrected — without it, testing five
things at α = 0.05 gives roughly a 1-in-4 chance of at least one false "significant".

In [ ]:
def rankdata(a):
    """Average ranks; ties share their mean rank (scipy's 'average' method)."""
    a = np.asarray(a, float)
    r = np.empty(len(a), float)
    r[a.argsort()] = np.arange(1, len(a) + 1)
    for v in np.unique(a[pd.Series(a).duplicated(keep=False).to_numpy()]):
        m = a == v
        r[m] = r[m].mean()
    return r


def spearman_r(u, v):
    return float(np.corrcoef(rankdata(u), rankdata(v))[0, 1])


def perm_null(ranks_x, ranks_y, n_perm=N_PERM, seed=SEED):
    """Rho under H0: x-ranks fixed, y-ranks shuffled.

    Permuting the observed y-ranks rather than 1..n keeps any tie structure intact,
    which matters wherever two generators share a median.
    """
    P = np.argsort(np.random.default_rng(seed).random((n_perm, len(ranks_x))), axis=1)
    a = ranks_x - ranks_x.mean()
    B = ranks_y[P] - ranks_y.mean()
    return (B @ a) / np.sqrt((a @ a) * (B ** 2).sum(axis=1))


def perm_p(rho, null):
    """Two-sided, +1-corrected so a p-value is never reported as exactly zero."""
    return (np.sum(np.abs(null) >= abs(rho) - 1e-12) + 1) / (len(null) + 1)


def holm(pvals):
    p, m = np.asarray(pvals, float), len(pvals)
    adj, run = np.empty(m), 0.0
    for k, i in enumerate(p.argsort()):
        run = max(run, (m - k) * p[i])
        adj[i] = min(run, 1.0)
    return adj

In [ ]:
rx      = rankdata(x)
Draw    = M[(M.label == 1) & (M.generator_family == "diffusion")]
rows, nulls = [], {}

for d in DETECTORS:
    # the sigma rescaling is monotone, so raw and sigma scores must give identical rho
    raw = Draw.groupby("generator")[d].median().reindex(g.generator).to_numpy()
    assert abs(spearman_r(x, raw) - spearman_r(x, g[d])) < 1e-12, f"{d}: rho not scale-invariant"

    ry        = rankdata(g[d].to_numpy())
    nulls[d]  = perm_null(rx, ry)
    r         = spearman_r(x, g[d])
    rows.append({"detector": zcols[d], "R": r, "ties": int(g[d].duplicated().sum()),
                 "p_perm": perm_p(r, nulls[d])})

R = pd.DataFrame(rows)
R["p_holm"]  = holm(R.p_perm)
R["verdict"] = np.where(R.p_holm < 0.05, "significant", "not distinguishable from chance")

print(f"permutation null: {N_PERM:,} shuffles per detector, p floor {1/(N_PERM+1):.1e}\n")
print(R.assign(R=R.R.round(3),
               p_perm=R.p_perm.map("{:.2g}".format),
               p_holm=R.p_holm.map("{:.2g}".format)).to_string(index=False))

### Reading the two figures

**Top figure** — the data as it actually sits: median score against release date. **Bottom
figure** — the same 16 generators in *rank* space, which is literally where Spearman lives. If
R were +1 every point would land on the dashed diagonal; if −1, on the anti-diagonal. Rank space
is also where a single freak generator stops being able to swing the answer.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(13.5, 5.4))

for j, d in enumerate(DETECTORS):
    r, p = R.R[j], R.p_perm[j]
    star = "*" if R.p_holm[j] < 0.05 else ""
    col  = FAKE if r < 0 else REAL
    ptxt = f"p<{1/(N_PERM+1):.0e}" if p <= 2/(N_PERM+1) else f"p={p:.3f}"

    ax = axes[0, j]
    ax.scatter(g.date, g[d], s=26, color=col, alpha=0.85, zorder=3,
               edgecolor="white", linewidth=0.6)
    ax.axhline(0, color=MUTED, lw=1, zorder=1)
    ax.set_title(f"{zcols[d]}\nR={r:+.2f}{star}  {ptxt}", fontsize=8)
    ax.tick_params(axis="x", labelrotation=45, labelsize=7)

    ax = axes[1, j]
    ax.plot([1, nG], [1, nG], ls="--", lw=1, color=MUTED, zorder=1)
    ax.plot([1, nG], [nG, 1], ls=":", lw=1, color=MUTED, zorder=1)
    ax.scatter(rankdata(x), rankdata(g[d]), s=26, color=col, alpha=0.85, zorder=3,
               edgecolor="white", linewidth=0.6)
    ax.set_xlabel("rank of release date", fontsize=7.5)
    ax.set_xlim(0.2, nG + 0.8); ax.set_ylim(0.2, nG + 0.8)
    ax.tick_params(labelsize=7)

axes[0, 0].set_ylabel("median score\n(σ above authentic)")
axes[1, 0].set_ylabel("rank of median score")
fig.suptitle("diffusion family only (n=16 generators)   ·   red = score falls with release date"
             "   ·   * = significant after Holm",
             x=0.005, ha="left", fontsize=10, fontweight="bold")
fig.tight_layout()
plt.show()

## 4. Sensitivity: what could this test have detected?

A non-significant result at n = 16 is ambiguous — it could mean "no temporal signal" or "there
is one but 16 points cannot see it." Distinguishing those is not optional, and the permutation
null answers it directly: it *is* the distribution of R when release date carries no information,
so its 95th percentile is the smallest |R| this design can call significant.

Below that value, a real trend and no trend are indistinguishable here. Any detector whose R is
inside the gray band should be read as **"this experiment cannot tell,"** not as "no effect."

In [ ]:
crit = {d: float(np.quantile(np.abs(nulls[d]), 0.95)) for d in DETECTORS}
crit_holm = {d: float(np.quantile(np.abs(nulls[d]), 1 - 0.05 / len(DETECTORS)))
             for d in DETECTORS}
crit95 = max(crit.values())   # most conservative of the five, used for the band below

print(f"n = {nG}   ->   |R| must exceed ~{crit95:.3f} to reach p<0.05 uncorrected")
print(f"                           and ~{max(crit_holm.values()):.3f} to survive Holm "
      f"across {len(DETECTORS)} detectors")
print(f"\nper-detector 95th pct of |R| under H0 (they differ only via ties):")
for d in DETECTORS:
    print(f"  {zcols[d]:<24} {crit[d]:.3f}")

order = np.argsort(R.R.to_numpy())
fig, ax = plt.subplots(figsize=(7.4, 2.9))

ax.axvspan(-crit95, crit95, color=NULLBAND, zorder=0)
ax.axvline(0, color=MUTED, lw=1, zorder=1)

for row, j in enumerate(order):
    r, sig = R.R[j], R.p_holm[j] < 0.05
    col = FAKE if r < 0 else REAL
    ax.plot([0, r], [row, row], color=col, lw=2, zorder=2, solid_capstyle="round")
    ax.scatter([r], [row], s=95, zorder=3, color=col if sig else "white",
               edgecolor=col, linewidth=1.6)
    ax.text(r + (0.055 if r >= 0 else -0.055), row,
            f"{r:+.2f}" + ("  significant" if sig else "  n.s."),
            va="center", ha="left" if r >= 0 else "right", fontsize=7.5, color="#333")

ax.set_yticks(range(len(order)))
ax.set_yticklabels([R.detector[j] for j in order], fontsize=8)
ax.set_xlabel("Spearman R   (median score vs release date, diffusion only)")
ax.set_xlim(-1.28, 1.28)
ax.set_ylim(-0.6, len(order) - 0.4)
ax.grid(axis="y", visible=False)
ax.set_title(f"gray band = where chance alone reaches |R| 95% of the time at n={nG}"
             f"   ·   filled dot = survives Holm")
fig.tight_layout()
plt.show()

## 5. Why the per-image version is not the answer

The same test with one row per **image**. Nothing new is measured — every image inherits its
generator's date — but n goes from 16 to 80,000 and the p-values collapse.

This is pseudo-replication, the same failure CLAUDE.md flags for the aggregator: *excess
temporal granularity biases crossings earlier than warranted — the dangerous direction*.
Reported only so the inflation is on the record.

In [ ]:
xi = D.release_date.map(pd.Timestamp.toordinal).to_numpy(float)
cmp = pd.DataFrame({
    "detector":       [zcols[d] for d in DETECTORS],
    "R_per_gen":      R.R.round(3),
    "p_per_gen":      R.p_perm.map("{:.2g}".format),
    "R_per_image":    [round(spearman_r(xi, D[d]), 3) for d in DETECTORS],
})
rxi = rankdata(xi)                                  # heavily tied: 16 dates, 272 images
cmp["p_per_image"] = [
    perm_p(r, perm_null(rxi, rankdata(D[d].to_numpy()), n_perm=20_000))
    for r, d in zip(cmp.R_per_image, DETECTORS)
]
cmp["p_per_image"] = cmp.p_per_image.map("{:.2g}".format)

print(f"per-generator n = {nG}      per-image n = {len(D)}   "
      f"(but still only {nG} distinct dates)\n")
print(cmp.to_string(index=False))

## Brief: what this settles, and what it does not

| detector | R | p | Holm | |
|---|---|---|---|---|
| univfd | **−0.785** | 0.00063 | **0.003** | significant |
| dmid_progan | −0.30 | 0.25 | 1 | cannot tell |
| cnndetection | −0.17 | 0.52 | 1 | cannot tell |
| aeroblade | −0.09 | 0.73 | 1 | cannot tell |
| dmid_latent | +0.14 | 0.60 | 1 | cannot tell |

- **Only UnivFD gets clearly worse on newer diffusion generators** (R = −0.785, still
  p = 0.003 after Holm). The frozen-CLIP probe was the member expected to generalise best.
  Stronger than on the local sample (−0.67, Holm 0.03), because 5,000-image medians are far less
  noisy than 17-image ones.
- **The other four are "cannot tell", not "no effect".** |R| must exceed 0.50 for p < 0.05 and
  0.64 to survive Holm. Those thresholds depend only on having 16 generators, so **more images
  will not change this — more generators will** (e.g. post-Aug-2024 models).
- **dmid_latent points the other way** (+0.14, n.s.), directionally consistent with it being the
  latent-diffusion-trained checkpoint. A hypothesis, not a finding.
- **Consequence for the architecture:** panel members age at different rates, so the
  calibration layer cannot treat detectors as exchangeable over time.
- **§5, the trap:** counted per image, four detectors hit the p floor (5e-5) — including
  dmid_latent's wrong-direction +0.15 — and AEROBLADE gets p = 0.0004 at R = −0.012. That is why the generator stays
  the unit of analysis.

Caveats:
- **UnivFD's trend leans on three late generators**: DeepFloyd IF (0.21 σ), DALL-E 3 (0.64) and
  FLUX 1 Dev (0.20). FLUX 1 Schnell (1.04) does not follow it. Five generators fall in a
  six-week window (Nov–Dec 2021), and the two FLUX models are a day apart. Shifting a few
  release dates by months could move R; testing that is a prerequisite for putting −0.785 in the
  thesis.
- **Date is still confounded with content and native resolution inside the family** (a 2020 DDPM
  sample and a 2024 FLUX sample differ in more than age). The 200×200 crop bounds the resolution
  part but does not remove it.
- The local caveat "17-image medians attenuate R" no longer applies at 5,000 per generator.